In [1]:
import json
from pathlib import Path

# Base folders
base_dir = Path("/Users/sterinsaji/Desktop/rag-research/notebooks/new_300_question_set")
src_dir = base_dir / "500_questions_from_each"
dst_dir = base_dir / "150_question_from_each"

# Source -> destination mapping (human labels intentionally excluded)
file_map = {
    "baseline_score.jsonl": "baseline_score_filtered.jsonl",
    "ragas_hallucination_results.jsonl": "ragas_hallucination_results_filtered.jsonl",
    "selfcheck_dataset.jsonl": "selfcheck_dataset_filtered.jsonl",
    "similarity_dataset.jsonl": "similarity_dataset_filtered.jsonl",
}

ADD_COUNT = 150

def read_jsonl(path: Path):
    rows = []
    if not path.exists():
        return rows

    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"[WARN] Skipping invalid JSON at {path.name}:{line_no}")
    return rows

def record_key(obj):
    # Prefer stable semantic key if present; otherwise canonical JSON text.
    if isinstance(obj, dict) and "id" in obj and obj["id"] is not None:
        return f"id::{obj['id']}"
    return "json::" + json.dumps(obj, ensure_ascii=False, sort_keys=True)

def append_jsonl(path: Path, rows):
    if not rows:
        return
    with path.open("a", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

summary = []

for src_name, dst_name in file_map.items():
    src_path = src_dir / src_name
    dst_path = dst_dir / dst_name

    if not src_path.exists():
        summary.append({
            "file": src_name,
            "status": "source_missing",
            "added": 0,
            "target": str(dst_path),
        })
        continue

    source_rows = read_jsonl(src_path)
    target_rows = read_jsonl(dst_path)

    existing_keys = {record_key(r) for r in target_rows}
    to_add = []

    for row in source_rows:
        k = record_key(row)
        if k in existing_keys:
            continue
        to_add.append(row)
        existing_keys.add(k)
        if len(to_add) == ADD_COUNT:
            break

    # Ensure destination exists and append
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    if not dst_path.exists():
        dst_path.touch()

    append_jsonl(dst_path, to_add)

    summary.append({
        "file": src_name,
        "target": dst_name,
        "existing_before": len(target_rows),
        "added": len(to_add),
        "final_count": len(target_rows) + len(to_add),
        "requested": ADD_COUNT,
        "status": "ok" if len(to_add) == ADD_COUNT else "added_less_than_requested",
    })

print("Done. Summary:")
for row in summary:
    print(row)

Done. Summary:
{'file': 'baseline_score.jsonl', 'target': 'baseline_score_filtered.jsonl', 'existing_before': 149, 'added': 150, 'final_count': 299, 'requested': 150, 'status': 'ok'}
{'file': 'ragas_hallucination_results.jsonl', 'target': 'ragas_hallucination_results_filtered.jsonl', 'existing_before': 150, 'added': 150, 'final_count': 300, 'requested': 150, 'status': 'ok'}
{'file': 'selfcheck_dataset.jsonl', 'target': 'selfcheck_dataset_filtered.jsonl', 'existing_before': 150, 'added': 150, 'final_count': 300, 'requested': 150, 'status': 'ok'}
{'file': 'similarity_dataset.jsonl', 'target': 'similarity_dataset_filtered.jsonl', 'existing_before': 150, 'added': 150, 'final_count': 300, 'requested': 150, 'status': 'ok'}


In [2]:
import json
from pathlib import Path

base_dir = Path("/Users/sterinsaji/Desktop/rag-research/notebooks/new_300_question_set")
baseline_path = base_dir / "150_question_from_each" / "baseline_score_filtered.jsonl"
human_label_path = base_dir / "150_question_from_each" / "human_label.jsonl"

TAKE_LAST_N = 150

def read_jsonl(path: Path):
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"[WARN] Invalid JSON skipped: {path.name}:{line_no}")
    return rows

def make_key(obj):
    if isinstance(obj, dict) and obj.get("id") is not None:
        return f"id::{obj['id']}"
    return "json::" + json.dumps(obj, ensure_ascii=False, sort_keys=True)

def drop_hallucination_key(obj):
    if not isinstance(obj, dict):
        return obj
    cleaned = dict(obj)
    cleaned.pop("hallucination", None)
    return cleaned

baseline_rows = read_jsonl(baseline_path)
human_rows = read_jsonl(human_label_path)

if len(baseline_rows) < TAKE_LAST_N:
    print(f"[WARN] baseline has only {len(baseline_rows)} rows; using all rows instead of last {TAKE_LAST_N}.")
    source_slice = baseline_rows
else:
    source_slice = baseline_rows[-TAKE_LAST_N:]

existing_keys = {make_key(r) for r in human_rows}
to_add = []

for row in source_slice:
    cleaned = drop_hallucination_key(row)
    k = make_key(cleaned)
    if k in existing_keys:
        continue
    to_add.append(cleaned)
    existing_keys.add(k)

human_label_path.parent.mkdir(parents=True, exist_ok=True)
if not human_label_path.exists():
    human_label_path.touch()

with human_label_path.open("a", encoding="utf-8") as f:
    for row in to_add:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Done.")
print(f"Baseline rows considered: {len(source_slice)}")
print(f"Rows appended to human_label.jsonl: {len(to_add)}")
print(f"human_label.jsonl final count: {len(human_rows) + len(to_add)}")

Done.
Baseline rows considered: 150
Rows appended to human_label.jsonl: 150
human_label.jsonl final count: 300


now I add this human label jsonl in flutter frontend to label it manually